<a href="https://colab.research.google.com/github/bangash-ds/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bangash-ds/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [4]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [6]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [7]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [8]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [10]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,hand_rule_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,0


# **My experiment**

In [14]:
trend_unique = (df['trend_direction'].value_counts()/len(df)*100).round(2)
print(trend_unique)

trend_direction
down      54.21
stable    19.87
up        14.63
new        7.45
flat       3.84
Name: count, dtype: float64


In [18]:
count_decline_label = (df['is_declining_label'].value_counts()/len(df)*100).round(2)
print(count_decline_label)

is_declining_label
1    54.21
0    45.79
Name: count, dtype: float64


# **Experiment 1**

In [20]:
from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np


In [21]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

for depth in [2, 3, 4]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    tree.fit(X, y)

    tree_score = tree.predict_proba(X)[:, 1]

    p20 = precision_at_k(tree_score, y, 20)
    p50 = precision_at_k(tree_score, y, 50)

    print(
        f"Depth {depth}: "
        f"Precision@20 = {p20:.3f}, "
        f"Precision@50 = {p50:.3f}"
    )

Depth 2: Precision@20 = 0.550, Precision@50 = 0.600
Depth 3: Precision@20 = 0.700, Precision@50 = 0.720
Depth 4: Precision@20 = 0.600, Precision@50 = 0.680


- Increasing the tree depth from 2 to 3 improved performance, with Precision@20 increasing from 0.550 to 0.700 and Precision@50 from 0.600 to 0.720.
- Increasing the depth further to 4 reduced performance compared with depth 3, suggesting that greater complexity did not provide additional useful signal in this run.
- The depth-3 tree achieved the best overall Precision@20 and Precision@50 among the tested tree depths.

# **Experiment 2 — Compare  hand rule with the trees**

In [22]:
print("Hand Rule:")
print(
    f"Precision@20 = "
    f"{precision_at_k(df['hand_rule_score'], y, 20):.3f}"
)

print(
    f"Precision@50 = "
    f"{precision_at_k(df['hand_rule_score'], y, 50):.3f}"
)

print("\nDecision Trees:")

for depth in [2, 3, 4]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    tree.fit(X, y)

    tree_score = tree.predict_proba(X)[:, 1]

    print(
        f"Depth {depth}: "
        f"Precision@20 = {precision_at_k(tree_score, y, 20):.3f}, "
        f"Precision@50 = {precision_at_k(tree_score, y, 50):.3f}"
    )

Hand Rule:
Precision@20 = 0.900
Precision@50 = 0.680

Decision Trees:
Depth 2: Precision@20 = 0.550, Precision@50 = 0.600
Depth 3: Precision@20 = 0.700, Precision@50 = 0.720
Depth 4: Precision@20 = 0.600, Precision@50 = 0.680


- The hand-written rule achieved the highest Precision@20 (0.900), outperforming all Decision Trees at the top 20 pages.
- The depth-3 Decision Tree achieved the highest Precision@50 (0.720), outperforming the hand rule (0.680) deeper in the ranking.
- This suggests that the simple hand rule is stronger at identifying the very highest-priority pages, while the depth-3 model captures additional signal when considering a larger set of pages.

# **Experiment 3 — See what the tree actually learned**

In [23]:
from sklearn.tree import export_text

tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



- The tree first splits on `impressions_90d`, making it the strongest splitting feature in this model.
- When `impressions_90d > 5.5`, the tree next considers `content_age_days`, showing that content age provides additional signal for this group.
- For newer pages (`content_age_days <= 312.5`), the tree uses `ctr` to make its final decision.
- For older pages (`content_age_days > 312.5`), the tree uses `avg_position` instead.
- The model mainly relies on `impressions_90d`, `content_age_days`, `ctr`, and `avg_position`; the remaining features were not used for splits in this tree.
- Because the tree has a limited depth of 3, its learned decision rules remain relatively easy to read and interpret.

# **Experiment 5 — Check feature importance**

In [24]:
importance = pd.Series(
    tree.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

impressions_90d           0.563911
content_age_days          0.251026
avg_position              0.114519
ctr                       0.070544
days_since_last_update    0.000000
word_count                0.000000
engagement_rate           0.000000
dtype: float64


- impressions_90d was the most important feature with an importance of approximately 0.563.
- content_age_days was the second most important feature with an importance of approximately 0.251.
- avg_position and ctr contributed smaller amounts, while days_since_last_update, word_count, and engagement_rate had zero importance in this tree.
- This indicates that the model relied primarily on page exposure and content age rather than all available features.

# **Conclusion**
- Overall, the depth-3 Decision Tree provided the best model performance in this experiment, while the hand-written rule remained strongest at Precision@20.
- The results show that increasing model complexity can improve performance up to a point, but a deeper model is not automatically better.
- The experiment also demonstrates the importance of interpreting the learned rules rather than evaluating the model only by its score.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.